# MathArena Structured v4 vs Free-Text Rubric Eval

This notebook runs the rubric benchmark loop we care about:

1. Load local MathArena USAMO 2026 joined rows.
2. Generate a structured v4 rubric from MathArena's source grading scheme.
3. Grade the same candidate solutions two ways:
   - **structured_v4**: candidate solution + generated v4 rubric -> judgment tree -> deterministic score
   - **free_text**: candidate solution + original free-text grading scheme -> model score
4. Compare both against MathArena's existing judge score (`points_judge_1`).

LLM calls are disabled by default. Set `RUN_MODEL_CALLS = True` to run the live benchmark.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'rubric_arena' / 'pipeline.py').exists():
            return candidate
    raise RuntimeError('Could not find rubric-arena repo root')

REPO_ROOT = find_repo_root(Path.cwd()).resolve()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
print('repo root:', REPO_ROOT)

from rubric_arena.matharena_loader import load_matharena_usamo_pipeline_rows
from rubric_arena.pipeline import (
    anthropic_text_call,
    build_final_score_rows,
    compare_to_ground_truth,
    flatten_all_structured_judgments,
    generate_structured_rubric,
    grade_free_text,
    grade_structured,
    holistic_vs_structured_diagnostics,
    safe_id,
    score_distribution_metrics,
    summarize_structured_atoms,
    write_jsonl,
)

DATA_ROOT = REPO_ROOT / 'data/matharena_usamo_2026'
RUBRIC_DIR = DATA_ROOT / 'rubrics'
RUN_DIR = DATA_ROOT / 'grading_runs'
RUBRIC_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
PROBLEM_IDX = 2
MODEL_NAME = 'Gemini 3.1 Pro Preview'
ROW_LIMIT = 4
GRADER_MODEL = 'claude-sonnet-4-6'
RUBRIC_MODEL = GRADER_MODEL
RUN_MODEL_CALLS = False
REUSE_RUBRIC = True


In [ ]:
rows = load_matharena_usamo_pipeline_rows(
    DATA_ROOT,
    problem_idx=PROBLEM_IDX,
    model_name=MODEL_NAME,
    limit=ROW_LIMIT,
)
print('rows:', len(rows))
assert rows, 'Run scripts/download_matharena_usamo_2026.py first or adjust filters'
print(rows[0]['problem_id'], rows[0]['model_name'], rows[0]['ground_truth_score'])
print(rows[0]['problem'][:500])


## Source Grading Scheme

This is the free-text rubric from MathArena. The structured-rubric path first translates this into v4 JSON. The baseline path passes this text directly to the grading LLM.


In [ ]:
print(rows[0]['grading_scheme'][:2500])


## Generate or Load Structured v4 Rubric

The generated rubric is cached locally so repeated grading runs can reuse the same rubric.


In [ ]:
rubric_path = RUBRIC_DIR / f"{rows[0]['problem_id']}.{safe_id(RUBRIC_MODEL)}.rubric.v4.json"
structured_rubric = None

if rubric_path.exists() and REUSE_RUBRIC:
    structured_rubric = json.loads(rubric_path.read_text())
    print('loaded cached rubric:', rubric_path)
elif RUN_MODEL_CALLS:
    assert os.environ.get('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY is required'
    rubric_call = anthropic_text_call(model=RUBRIC_MODEL, temperature=0.0, max_tokens=8192)
    generated = generate_structured_rubric(rows[0], llm_call=rubric_call, rubric_model=RUBRIC_MODEL)
    structured_rubric = generated['rubric']
    rubric_path.write_text(json.dumps(structured_rubric, indent=2, ensure_ascii=False) + '\n')
    rubric_path.with_suffix('.generation.json').write_text(json.dumps(generated, indent=2, ensure_ascii=False) + '\n')
    print('wrote', rubric_path)
else:
    print('model calls disabled and no cached rubric exists:', rubric_path)

if structured_rubric:
    print(json.dumps(structured_rubric, indent=2, ensure_ascii=False)[:2500])


## Run Structured-vs-Free-Text Grading

This grades the same MathArena candidate solutions with both methods and writes JSONL results locally.


In [ ]:
results = []

if RUN_MODEL_CALLS:
    assert os.environ.get('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY is required'
    grader_call = anthropic_text_call(model=GRADER_MODEL, temperature=0.0, max_tokens=8192)
    if structured_rubric is None:
        raise RuntimeError('structured_rubric is required for structured_v4 grading')

    for row in rows:
        structured_result = grade_structured(
            row,
            rubric=structured_rubric,
            llm_call=grader_call,
            grader_model=GRADER_MODEL,
        )
        free_text_result = grade_free_text(
            row,
            llm_call=grader_call,
            grader_model=GRADER_MODEL,
        )
        results.extend([structured_result, free_text_result])
        print(
            row['id'],
            'gt=', row.get('ground_truth_score'),
            'structured=', structured_result.get('computed_score'),
            'free_text=', free_text_result.get('computed_score'),
        )

    out_path = RUN_DIR / f"p{PROBLEM_IDX}.{safe_id(MODEL_NAME)}.{safe_id(GRADER_MODEL)}.structured_vs_free_text.jsonl"
    write_jsonl(out_path, results)
    final_score_rows = build_final_score_rows(results)
    atom_rows = flatten_all_structured_judgments(results)
    paired_rows = holistic_vs_structured_diagnostics(results)
    metrics = compare_to_ground_truth(results)
    metrics['score_distributions'] = score_distribution_metrics(results)
    metrics['structured_atoms'] = summarize_structured_atoms(atom_rows)

    write_jsonl(out_path.with_suffix('.final_scores.jsonl'), final_score_rows)
    write_jsonl(out_path.with_suffix('.structured_atoms.jsonl'), atom_rows)
    write_jsonl(out_path.with_suffix('.paired_diagnostics.jsonl'), paired_rows)
    out_path.with_suffix('.metrics.json').write_text(json.dumps(metrics, indent=2, ensure_ascii=False) + '\n')
    print('wrote', out_path)
    print(json.dumps(metrics, indent=2, ensure_ascii=False))
else:
    print('model calls disabled')


## Inspect Existing Results

Use this after running the live grading cells or the CLI script.


In [ ]:
existing = []
for path in sorted(RUN_DIR.glob('*.jsonl')):
    with path.open() as f:
        for line in f:
            if line.strip():
                existing.append(json.loads(line))
print('existing graded rows:', len(existing))
if existing:
    print(json.dumps(compare_to_ground_truth(existing), indent=2, ensure_ascii=False))


## Final-Score and Atom-Level Diagnostics

These tables are the part of the experiment that goes beyond holistic score matching. The final-score table compares free-text and structured scores against MathArena. The atom table explodes structured judgments into per-node decisions, which is what we need for localized error analysis, disagreement rates, and TVD-like metrics over binary tasks.


In [ ]:
existing = []
for path in sorted(RUN_DIR.glob('*.jsonl')):
    if any(suffix in path.name for suffix in ['final_scores', 'structured_atoms', 'paired_diagnostics']):
        continue
    with path.open() as f:
        for line in f:
            if line.strip():
                existing.append(json.loads(line))

print('existing raw grading rows:', len(existing))
if existing:
    final_score_rows = build_final_score_rows(existing)
    atom_rows = flatten_all_structured_judgments(existing)
    paired_rows = holistic_vs_structured_diagnostics(existing)
    metrics = compare_to_ground_truth(existing)
    metrics['score_distributions'] = score_distribution_metrics(existing)
    metrics['structured_atoms'] = summarize_structured_atoms(atom_rows)
    print('final score rows:', len(final_score_rows))
    print('structured atom rows:', len(atom_rows))
    print('paired rows:', len(paired_rows))
    print(json.dumps(metrics, indent=2, ensure_ascii=False)[:4000])
else:
    print('No grading result JSONL files found yet.')


## Interpreting the Experiment

The headline metric is whether `structured_v4` matches or beats `free_text` on final-score error. The process-grading value comes from the atom table: even when final scores tie, structured grading tells us which regime was selected, which binary conditions were satisfied, and where repeated graders disagree. That is the data surface we need for debate and for higher-ESS analysis over rubric atoms rather than whole-problem scores.


## CLI Equivalent

The notebook path and CLI path use the same code:

```bash
uv run python scripts/run_matharena_eval.py   --problem-idx 2   --model-name "Gemini 3.1 Pro Preview"   --limit 4   --grader-model claude-sonnet-4-6   --method both   --reuse-rubric
```

The CLI writes raw results plus:

```text
*.metrics.json
*.final_scores.jsonl
*.structured_atoms.jsonl
*.paired_diagnostics.jsonl
```
